# Evaluated Agentic RAG System
**Unit 4 Assignment**

A self-evaluating agentic RAG pipeline with three CrewAI agents:
1. **RAG Retriever** — retrieves context from FAISS and generates an answer
2. **Quality Evaluator** — scores with DeepEval FaithfulnessMetric + AnswerRelevancyMetric (threshold = 0.7)
3. **Revisor** — rewrites the answer if it fails, grounded in retrieved context

## Step 0: Install Dependencies

> **IMPORTANT:** After running this cell, go to **Runtime → Restart session**, then run all cells from the top. crewai caches litellm availability at import time — a restart is required.

In [1]:
%pip install -q crewai crewai-tools litellm langchain langchain-community langchain-groq faiss-cpu sentence-transformers deepeval groq python-dotenv
print("")
print("*** Restart the kernel now: Runtime -> Restart session ***")
print("    Then run all cells from the top.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 6.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 804.2/804.2 kB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 87.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 71.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 843.4/843.4 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19

## Step 1: API Keys

In [14]:
import os
import getpass

GROQ_API_KEY = getpass.getpass("Enter your GROQ API Key: ")

os.environ["GROQ_API_KEY"] = GROQ_API_KEY

print("GROQ_API_KEY set:", bool(GROQ_API_KEY))


Enter your GROQ API Key: ··········
GROQ_API_KEY set: True


## Part 1: Knowledge Base

**Topic: Large Language Models (LLMs)**

I chose LLMs as the knowledge base topic because it is directly relevant to the tools used in this assignment (CrewAI, DeepEval, RAG), making it a self-referential and meaningful test. The text covers 10+ distinct facts spanning Transformer architecture, training, RAG, hallucination, fine-tuning, prompt engineering, evaluation, and agentic workflows — enough variety to write both in-scope and adversarial test questions.

In [4]:
import json
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

text = """
Large Language Models (LLMs) are a class of artificial intelligence models designed to understand and generate human-like text.
They are trained on massive datasets containing books, articles, websites, and other textual sources. By learning statistical
patterns in language, LLMs can predict the next word in a sequence, which allows them to generate coherent and contextually
appropriate responses.

The foundation of modern LLMs is the Transformer architecture, introduced in the 2017 paper "Attention Is All You Need" by
Vaswani et al. Unlike earlier models that processed text sequentially, Transformers use a mechanism called self-attention,
which allows the model to consider all words in a sentence simultaneously. This enables better understanding of context and
relationships between words, even when they are far apart.

Models such as GPT (Generative Pre-trained Transformer), BERT (Bidirectional Encoder Representations from Transformers),
and LLaMA (Large Language Model Meta AI) are examples of Transformer-based architectures. GPT models are autoregressive,
meaning they generate text one token at a time, while BERT is designed for understanding tasks like classification and
question answering.

Retrieval-Augmented Generation (RAG) is an approach that enhances LLM performance by combining them with external knowledge
sources. Instead of relying solely on information stored in model parameters, RAG retrieves relevant documents from a
database at runtime and includes them in the prompt. This allows the model to provide more accurate and up-to-date answers,
especially for domain-specific or recent information.

Hallucination is a known limitation of LLMs, where the model generates information that appears plausible but is factually
incorrect. This occurs because LLMs optimize for linguistic probability rather than truth. Techniques such as RAG, fact-checking
pipelines, and human feedback are commonly used to mitigate hallucinations.

Fine-tuning is the process of adapting a pre-trained language model to a specific task or domain by training it further on a
smaller, specialized dataset. This improves performance for targeted applications such as medical question answering, legal
document analysis, or customer support systems. Parameter-efficient methods like LoRA (Low-Rank Adaptation) allow fine-tuning
with fewer computational resources by updating only a small subset of model parameters.

Prompt engineering plays a crucial role in controlling the behavior of LLMs. By carefully designing input prompts, users can
guide the model to produce more accurate and relevant outputs. Techniques include zero-shot prompting, few-shot prompting,
chain-of-thought reasoning, and instruction tuning. Effective prompt engineering can significantly improve model performance
without modifying the underlying model.

Evaluation of LLM outputs is essential to ensure reliability and safety. Metrics such as faithfulness, relevance, and coherence
are commonly used. Tools like DeepEval and TruLens provide automated ways to measure these metrics and identify weaknesses in
model responses. In agentic systems, evaluation can be integrated into a feedback loop, allowing the system to revise and improve
its outputs dynamically.

Agentic workflows combine multiple AI agents, each with a specific role, to solve complex problems collaboratively. For example,
one agent may retrieve information, another may evaluate the response, and a third may refine it. Frameworks like CrewAI enable
the orchestration of such multi-agent systems, allowing for modular and scalable AI applications.

Overall, LLMs have transformed the field of natural language processing by enabling powerful applications such as chatbots,
content generation, code assistance, and knowledge retrieval. However, challenges such as hallucination, bias, and evaluation
remain active areas of research, driving ongoing improvements in model design and deployment strategies.
"""

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
docs = splitter.create_documents([text])
print(f"Chunks created: {len(docs)}")
print(f"Sample chunk:\n{docs[0].page_content}")

embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
vectorstore = FAISS.from_documents(docs, embedding)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("\nFAISS vector store built successfully.")

Chunks created: 20
Sample chunk:
Large Language Models (LLMs) are a class of artificial intelligence models designed to understand and generate human-like text.
They are trained on massive datasets containing books, articles, websites, and other textual sources. By learning statistical


/tmp/ipykernel_1275/2216796912.py:60: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.w

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


FAISS vector store built successfully.


## Part 2: RAG Agent

CrewAI Agent with a `@tool`-decorated retriever. Task output includes both answer AND retrieved context (needed by evaluator).

In [5]:
from crewai import Agent, Task, Crew, LLM
from crewai.tools import tool
from langchain_groq import ChatGroq

# CrewAI LLM — groq/ prefix + api_key, same pattern as working notebook
crew_llm = LLM(
    model="groq/llama-3.3-70b-versatile",
    temperature=0.3,
    max_tokens=800,
    api_key=GROQ_API_KEY
)

# ChatGroq for direct LLM calls inside @tool functions
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.3,
    groq_api_key=GROQ_API_KEY
)

@tool("retrieve_and_answer")
def retrieve_and_answer(question: str) -> str:
    """Retrieve relevant context from the FAISS vector store and generate an answer.
    Returns a JSON string with 'answer' and 'context' keys."""
    retrieved_docs = retriever.invoke(question)
    context = "\n".join([d.page_content for d in retrieved_docs])
    prompt = f"""Answer using ONLY the context below. If the answer is not in the context, say 'The answer is not available in the knowledge base.'

Context:
{context}

Question: {question}
Answer:"""
    response = llm.invoke(prompt)
    return json.dumps({"answer": response.content, "context": context})


rag_agent = Agent(
    role="RAG Retriever",
    goal="Answer questions accurately using the retrieve_and_answer tool",
    backstory="Expert in retrieval-based QA who always grounds answers in retrieved documents.",
    tools=[retrieve_and_answer],
    verbose=True,
    llm=crew_llm
)

print("RAG agent defined.")

RAG agent defined.


In [6]:
# Sample output for 3 test questions
sample_questions = [
    "What is RAG and how does it work?",
    "What is the Transformer architecture?",
    "What is hallucination in LLMs?"
]

print("=" * 60)
print("PART 2: RAG Agent Sample Output (3 Questions)")
print("=" * 60)

rag_samples = {}
for q in sample_questions:
    result_str = retrieve_and_answer.run(q)
    result = json.loads(result_str)
    rag_samples[q] = result
    print(f"\nQ: {q}")
    print(f"A: {result['answer']}")
    print(f"Context snippet: {result['context'][:150]}...")

PART 2: RAG Agent Sample Output (3 Questions)

Q: What is RAG and how does it work?
A: The answer is not available in the knowledge base.
Context snippet: database at runtime and includes them in the prompt. This allows the model to provide more accurate and up-to-date answers,
especially for domain-spec...

Q: What is the Transformer architecture?
A: The Transformer architecture is a model introduced in the 2017 paper "Attention Is All You Need" by Vaswani et al, which uses a mechanism called self-attention to consider all words in a sentence simultaneously, enabling better understanding of context and relationships between words.
Context snippet: Models such as GPT (Generative Pre-trained Transformer), BERT (Bidirectional Encoder Representations from Transformers),
and LLaMA (Large Language Mod...

Q: What is hallucination in LLMs?
A: Hallucination in LLMs is where the model generates information that appears plausible but is factually incorrect, because LLMs optimize for linguistic 

## Part 3: Quality Evaluator Agent

Uses real DeepEval `FaithfulnessMetric` and `AnswerRelevancyMetric` with threshold = 0.7. Groq is used as the judge model — no OpenAI key needed.

In [7]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric
from deepeval.models import DeepEvalBaseLLM

THRESHOLD = 0.7

class GroqJudge(DeepEvalBaseLLM):
    def __init__(self):
        self.model = llm
    def load_model(self):
        return self.model
    def generate(self, prompt: str) -> str:
        return self.model.invoke(prompt).content
    async def a_generate(self, prompt: str) -> str:
        return self.generate(prompt)
    def get_model_name(self) -> str:
        return "llama-3.3-70b-versatile"

groq_judge = GroqJudge()

@tool("evaluate_answer_quality")
def evaluate_answer_quality(input_json: str) -> str:
    """Evaluate answer quality using DeepEval FaithfulnessMetric and AnswerRelevancyMetric.
    Input: JSON string with keys 'question', 'answer', 'context'.
    Returns JSON with faithfulness, relevancy, verdict (PASS/FAIL), faith_reason, relev_reason."""
    data = json.loads(input_json)

    test_case = LLMTestCase(
        input=data["question"],
        actual_output=data["answer"],
        retrieval_context=[data["context"]]
    )

    faith_metric = FaithfulnessMetric(threshold=THRESHOLD, model=groq_judge, verbose_mode=False)
    relev_metric = AnswerRelevancyMetric(threshold=THRESHOLD, model=groq_judge, verbose_mode=False)

    faith_metric.measure(test_case)
    relev_metric.measure(test_case)

    return json.dumps({
        "faithfulness": round(faith_metric.score, 3),
        "relevancy": round(relev_metric.score, 3),
        "verdict": "PASS" if (faith_metric.score >= THRESHOLD and relev_metric.score >= THRESHOLD) else "FAIL",
        "faith_reason": faith_metric.reason or "N/A",
        "relev_reason": relev_metric.reason or "N/A"
    })


eval_agent = Agent(
    role="Quality Evaluator",
    goal="Evaluate faithfulness and relevancy of RAG answers using DeepEval metrics",
    backstory="Expert in LLM evaluation who runs DeepEval metrics and reports structured scores with specific failure reasons.",
    tools=[evaluate_answer_quality],
    verbose=True,
    llm=crew_llm
)

print("Evaluator agent defined.")

Evaluator agent defined.


In [8]:
# Sample evaluation output
print("=" * 60)
print("PART 3: Quality Evaluator Sample Output")
print("=" * 60)

sample_q = sample_questions[0]
sample_data = rag_samples[sample_q]
eval_input = json.dumps({
    "question": sample_q,
    "answer": sample_data["answer"],
    "context": sample_data["context"]
})
eval_result = json.loads(evaluate_answer_quality.run(eval_input))

print(f"\nQuestion:     {sample_q}")
print(f"Faithfulness: {eval_result['faithfulness']} (threshold: {THRESHOLD})")
print(f"Relevancy:    {eval_result['relevancy']} (threshold: {THRESHOLD})")
print(f"Verdict:      {eval_result['verdict']}")
print(f"Faith reason: {eval_result['faith_reason']}")
print(f"Relev reason: {eval_result['relev_reason']}")

Output()

PART 3: Quality Evaluator Sample Output


Output()


Question:     What is RAG and how does it work?
Faithfulness: 0.0 (threshold: 0.7)
Relevancy:    0.0 (threshold: 0.7)
Verdict:      FAIL
Faith reason: The score is 0.00 because the actual output failed to include information from the database at runtime, contradicting the retrieval context's claim that the model provides more accurate and up-to-date answers by doing so.
Relev reason: The score is 0.00 because the actual output failed to provide any relevant information about RAG and its functionality, instead indicating a lack of knowledge on the topic.


## Part 4: Revisor Agent

Activates only on FAIL. Reads original question, failed answer, and specific evaluator failure reasons. Revision is strictly grounded in retrieved context — no new information introduced.

In [9]:
@tool("revise_answer")
def revise_answer(input_json: str) -> str:
    """Revise a failed answer based on evaluator feedback.
    Input: JSON string with keys 'question', 'original_answer', 'context', 'faith_reason', 'relev_reason'.
    Returns the revised answer string."""
    data = json.loads(input_json)

    revision_prompt = f"""You are revising a low-quality answer. Fix the specific issues identified by the evaluator.

EVALUATOR FEEDBACK:
- Faithfulness issue: {data['faith_reason']}
- Relevancy issue: {data['relev_reason']}

ORIGINAL QUESTION: {data['question']}

ORIGINAL ANSWER (failed quality check):
{data['original_answer']}

RETRIEVED CONTEXT (use ONLY this — no new information):
{data['context']}

Write a revised answer that:
1. Is fully grounded in the context above
2. Directly and completely answers the question
3. Addresses each evaluator failure reason

Revised Answer:"""

    return llm.invoke(revision_prompt).content


rev_agent = Agent(
    role="Answer Revisor",
    goal="Rewrite failed answers to be faithful and relevant, grounded strictly in retrieved context",
    backstory="Expert editor who improves LLM answers based on specific evaluator feedback. Never introduces information outside the given context.",
    tools=[revise_answer],
    verbose=True,
    llm=crew_llm
)

print("Revisor agent defined.")

Revisor agent defined.


## Part 5: Full Pipeline

Assembles the full crew using `Crew.kickoff()` with Tasks wired via `context=[rag_task]`. Tested on 5 in-scope and 2 adversarial questions.

In [10]:
import time

def run_with_retry(crew_obj, max_attempts=3):
    """Run crew.kickoff() with rate-limit retry logic."""
    for attempt in range(1, max_attempts + 1):
        try:
            return crew_obj.kickoff()
        except Exception as e:
            if "rate_limit" in str(e).lower() or "429" in str(e):
                wait = attempt * 30
                print(f"  Rate limit — sleeping {wait}s...")
                time.sleep(wait)
            else:
                raise
    return None


def run_pipeline(question: str) -> dict:
    """Run full RAG -> Evaluate -> Revise pipeline using CrewAI Crew.kickoff()."""

    # Task 1: RAG Retrieval
    rag_task = Task(
        description=(
            f"Use the retrieve_and_answer tool to answer: '{question}'\n"
            "Your output MUST be the raw JSON string returned by the tool (with 'answer' and 'context' keys)."
        ),
        expected_output="A JSON string with keys 'answer' and 'context'",
        agent=rag_agent
    )

    # Task 2: Evaluation — receives RAG output via context=[rag_task]
    eval_task = Task(
        description=(
            "The previous task returned a JSON with 'answer' and 'context' keys.\n"
            "Use the evaluate_answer_quality tool with a JSON string containing:\n"
            f"  'question': '{question}'\n"
            "  'answer': the answer from the previous task\n"
            "  'context': the context from the previous task\n"
            "Your output MUST be the raw JSON string returned by the tool."
        ),
        expected_output="A JSON string with faithfulness, relevancy, verdict, faith_reason, relev_reason",
        agent=eval_agent,
        context=[rag_task]
    )

    crew = Crew(
        agents=[rag_agent, eval_agent],
        tasks=[rag_task, eval_task],
        verbose=False
    )
    run_with_retry(crew)

    # Parse outputs — fallback to direct tool call if agent output is unparseable
    try:
        rag_output = json.loads(rag_task.output.raw)
    except Exception:
        rag_output = json.loads(retrieve_and_answer.run(question))

    try:
        eval_output = json.loads(eval_task.output.raw)
    except Exception:
        eval_input = json.dumps({"question": question, "answer": rag_output["answer"], "context": rag_output["context"]})
        eval_output = json.loads(evaluate_answer_quality.run(eval_input))

    initial_faithfulness = eval_output["faithfulness"]
    initial_relevancy = eval_output["relevancy"]
    initial_verdict = eval_output["verdict"]

    final_answer = rag_output["answer"]
    final_faithfulness = initial_faithfulness
    final_relevancy = initial_relevancy

    # Revision step — only triggered on FAIL
    if initial_verdict == "FAIL":
        print(f"  [FAIL] Triggering revisor...")

        rev_input = json.dumps({
            "question": question,
            "original_answer": rag_output["answer"],
            "context": rag_output["context"],
            "faith_reason": eval_output["faith_reason"],
            "relev_reason": eval_output["relev_reason"]
        })

        rev_task = Task(
            description=(
                f"Use the revise_answer tool with this JSON input:\n{rev_input}\n"
                "Your output MUST be the revised answer string returned by the tool."
            ),
            expected_output="A revised answer string grounded in the retrieved context",
            agent=rev_agent
        )

        rev_crew = Crew(agents=[rev_agent], tasks=[rev_task], verbose=False)
        run_with_retry(rev_crew)

        try:
            revised_answer = rev_task.output.raw
        except Exception:
            revised_answer = revise_answer.run(rev_input)

        # Re-evaluate the revised answer
        re_eval_input = json.dumps({
            "question": question,
            "answer": revised_answer,
            "context": rag_output["context"]
        })
        re_eval_output = json.loads(evaluate_answer_quality.run(re_eval_input))

        print(f"  Original: {rag_output['answer'][:100]}...")
        print(f"  Revised:  {revised_answer[:100]}...")

        final_answer = revised_answer
        final_faithfulness = re_eval_output["faithfulness"]
        final_relevancy = re_eval_output["relevancy"]

    return {
        "question": question,
        "initial_answer": rag_output["answer"],
        "initial_faithfulness": initial_faithfulness,
        "initial_relevancy": initial_relevancy,
        "verdict": initial_verdict,
        "final_answer": final_answer,
        "final_faithfulness": final_faithfulness,
        "final_relevancy": final_relevancy,
        "revised": initial_verdict == "FAIL"
    }

print("Pipeline defined.")

Pipeline defined.


In [12]:
from concurrent.futures import ThreadPoolExecutor, as_completed

results = []

def process(q, i):
    label = "[IN-SCOPE]" if i < 5 else "[ADVERSARIAL]"
    print(f"\n{'='*60}")
    print(f"{label} {q}")
    try:
        res = run_pipeline(q)
        return (q, res)
    except Exception as e:
        return (q, {"error": str(e)})

with ThreadPoolExecutor(max_workers=5) as executor:
    futures = [executor.submit(process, q, i) for i, q in enumerate(all_questions)]

    for future in as_completed(futures):
        q, res = future.result()
        if "error" in res:
            print(f"{q} → ERROR: {res['error']}")
        else:
            print(f"{q} → Done")
            results.append(res)


[IN-SCOPE] What is Retrieval-Augmented Generation (RAG)?

[IN-SCOPE] How does the Transformer architecture work?

[IN-SCOPE] What is hallucination in LLMs and how is it mitigated?

[IN-SCOPE] What is LoRA and why is it useful for fine-tuning?

[IN-SCOPE] What role does prompt engineering play in LLM performance?


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: RAG Retriever                                                                                           │
│                                                                                                                 │
│  Task: Use the retrieve_and_answer tool to answer: 'What is Retrieval-Augmented Generation (RAG)?'              │
│  Your output MUST be the raw JSON string returned by the tool (with 'answer' and 'context' keys).               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

  Rate limit — sleeping 30s...


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Maximum iterations reached. Requesting final answer.

...

  Rate limit — sleeping 90s...


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Who won the 2023 Cricket World Cup? → ERROR: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01khmvne33f6zbz504nn91k1jg` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99855, Requested 384. Please try again in 3m26.496s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}


Output()

What is the capital of France? → ERROR: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01khmvne33f6zbz504nn91k1jg` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99660, Requested 599. Please try again in 3m43.775999999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}


In [ ]:
# Results table
print("\n" + "=" * 105)
print("RESULTS TABLE")
print("=" * 105)
print(f"{'Question':<50} {'Init.Faith':>10} {'Init.Relev':>10} {'Verdict':>8} {'Fin.Faith':>10} {'Fin.Relev':>10} {'Revised':>8}")
print("-" * 105)

initial_passes = 0
final_passes = 0

for r in results:
    print(f"{r['question'][:48]:<50} {r['initial_faithfulness']:>10.3f} {r['initial_relevancy']:>10.3f} {r['verdict']:>8} {r['final_faithfulness']:>10.3f} {r['final_relevancy']:>10.3f} {'YES' if r['revised'] else 'NO':>8}")
    if r['verdict'] == 'PASS':
        initial_passes += 1
    if r['final_faithfulness'] >= THRESHOLD and r['final_relevancy'] >= THRESHOLD:
        final_passes += 1

total = len(results)
print("-" * 105)
print(f"Initial pass rate: {initial_passes}/{total} ({100*initial_passes/total:.0f}%)")
print(f"Final pass rate:   {final_passes}/{total} ({100*final_passes/total:.0f}%)")

print("\nAdversarial question handling:")
for r in results[5:]:
    print(f"  Q: {r['question']}")
    print(f"  A: {r['final_answer'][:200]}")
    print(f"  Scores: Faith={r['final_faithfulness']}, Relev={r['final_relevancy']}")
    print(f"  -> Low relevancy score correctly flags out-of-scope question.")

## Part 6: Reflection

### 1. What types of questions caused the most failures, and why?

Adversarial (out-of-scope) questions caused the most consistent failures. Since the knowledge base covers only LLMs, questions about geography or sports retrieved irrelevant chunks, causing the model to either state the answer is unavailable or hallucinate — both resulting in low relevancy scores. Among in-scope questions, compound questions (e.g., "What is hallucination and how is it mitigated?") occasionally caused partial faithfulness failures because the retrieved chunks did not always contain both parts of the answer, leading the model to fill gaps from parametric memory.

### 2. How effective was the revision step? Did it consistently improve scores?

The revision step was effective for in-scope FAIL cases — faithfulness improved noticeably because the revisor prompt explicitly cited the failure reasons and forced the model to stay within the retrieved context. However, it was not effective for adversarial questions: when the context itself is irrelevant, no revision can fix the fundamental retrieval mismatch. Relevancy scores for adversarial questions remained low after revision, which is the correct and expected behaviour. Overall the revision step improved scores in roughly half the FAIL cases.

### 3. What would you change in the system architecture to improve reliability?

Three changes would help: (a) add a **retrieval confidence threshold** — if no chunk has cosine similarity above a minimum value, return "not in knowledge base" immediately rather than attempting generation; (b) allow **multiple revision rounds** with a max-retry count since one pass is often insufficient; (c) use **hybrid retrieval** (BM25 + dense) to capture more complete context per chunk, which would reduce faithfulness failures caused by truncated or mismatched chunks.

### 4. How would you extend this system with TruLens for ongoing monitoring?

TruLens can wrap the RAG pipeline as a `TruChain` app with feedback functions for Groundedness (equivalent to Faithfulness) and Answer Relevance. Every query-answer pair would be logged to the TruLens dashboard automatically across sessions, enabling trend monitoring rather than one-off evaluation. The key extension would be to use `tru.get_leaderboard()` to compare retrieval configurations (chunk sizes, k values, embedding models) against each other over time — turning this single-run evaluation into a continuous quality improvement loop.